# Financial Transaction Volume Forecasting with Google TimesFM

This notebook implements an end-to-end **transaction volume forecasting** pipeline for a FinTech payment platform.

## Problem

Payment systems process high transaction volume continuously. Underestimating future volume risks latency spikes and downtime; overestimating wastes infrastructure and analyst capacity.

## Input

- `Timestamp`
- `Transactions`

## Forecast Goal

Predict future transaction counts for:

- next hour
- next day
- next week

Then translate forecast outputs into:

- capacity planning
- resource allocation (fraud monitoring)
- reliability safeguards


## What You Will Learn

1. Download financial transaction data directly from web (Kaggle).
2. Build hourly transaction volume and fraud-load series from 1M records.
3. Apply data quality checks and regularize time index.
4. Train/evaluate TimesFM with calendar + fraud-rate covariates.
5. Benchmark against classical baselines.
6. Convert forecasts into production operations plans.


In [ ]:
from __future__ import annotations

import math
import os
import subprocess
import zipfile
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42
np.random.seed(SEED)

# Keep behavior robust in environments without GPU/JAX CUDA support.
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '')
os.environ.setdefault('JAX_PLATFORMS', 'cpu')

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / 'data' / 'financial_transactions'
RAW_DIR = DATA_DIR / 'raw'
ART_DIR = PROJECT_ROOT / 'artifacts' / 'financial_transaction_volume_timesfm'

RAW_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)

TXN_CSV = RAW_DIR / 'transactions.csv'

print('Project root:', PROJECT_ROOT)
print('Transaction CSV:', TXN_CSV)
print('Artifacts dir:', ART_DIR)

## 1) Download Data from Web

Dataset (Kaggle):

- `sergionefedov/fraud-detection-1m-transactions-7-fraud-types`

Why this dataset is suitable:

- 1M transaction-level events
- explicit timestamps
- fraud labels for fraud-monitoring load features


In [ ]:
def ensure_dataset(transactions_csv: Path) -> Path:
    if transactions_csv.exists():
        print(f'Found existing file: {transactions_csv}')
        return transactions_csv

    zip_path = RAW_DIR / 'fraud-detection-1m-transactions-7-fraud-types.zip'
    cmd = [
        'kaggle',
        'datasets',
        'download',
        '-d',
        'sergionefedov/fraud-detection-1m-transactions-7-fraud-types',
        '-p',
        str(RAW_DIR),
        '--force',
    ]

    try:
        print('Downloading dataset from Kaggle...')
        subprocess.run(cmd, check=True, capture_output=True, text=True)
    except Exception as exc:
        raise RuntimeError(
            'Failed to download dataset from Kaggle. Configure Kaggle API credentials or place '
            f'transactions.csv at {transactions_csv}.'
        ) from exc

    if zip_path.exists():
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(RAW_DIR)

    if not transactions_csv.exists():
        raise FileNotFoundError(f'Expected file not found after download: {transactions_csv}')

    return transactions_csv


txn_csv = ensure_dataset(TXN_CSV)
txn_csv

## 2) Build Hourly Transaction Volume Series

We load only required fields for speed/memory:

- `timestamp`
- `is_fraud`

Then aggregate hourly:

- `transactions`: total transaction count per hour
- `fraud_count`: fraud transaction count per hour
- `fraud_rate`: fraud_count / transactions


In [ ]:
usecols = ['timestamp', 'is_fraud']
raw_txn = pd.read_csv(txn_csv, usecols=usecols, parse_dates=['timestamp'])

print('Raw rows:', len(raw_txn))
print('Timestamp nulls:', int(raw_txn['timestamp'].isna().sum()))
print('Date range:', raw_txn['timestamp'].min(), '->', raw_txn['timestamp'].max())

raw_txn = raw_txn.dropna(subset=['timestamp']).copy()
raw_txn['is_fraud'] = pd.to_numeric(raw_txn['is_fraud'], errors='coerce').fillna(0.0).astype(np.float32)

hourly_df = (
    raw_txn.set_index('timestamp')
    .resample('h')
    .agg(
        transactions=('is_fraud', 'size'),
        fraud_count=('is_fraud', 'sum'),
    )
    .fillna(0.0)
    .reset_index()
)

hourly_df['fraud_rate'] = (hourly_df['fraud_count'] / hourly_df['transactions'].clip(lower=1)).astype(np.float32)
hourly_df[['transactions', 'fraud_count']] = hourly_df[['transactions', 'fraud_count']].astype(np.float32)

print('Hourly rows:', len(hourly_df))
print('Hourly range:', hourly_df['timestamp'].min(), '->', hourly_df['timestamp'].max())
print('Mean tx/hour:', float(hourly_df['transactions'].mean()))
print('Max tx/hour:', float(hourly_df['transactions'].max()))

hourly_df.head()

In [ ]:
# Enforce strict hourly grid.
full_hours = pd.date_range(hourly_df['timestamp'].min(), hourly_df['timestamp'].max(), freq='h')
series_df = (
    hourly_df.set_index('timestamp')
    .reindex(full_hours)
    .fillna(0.0)
    .rename_axis('timestamp')
    .reset_index()
)

for col in ['transactions', 'fraud_count', 'fraud_rate']:
    series_df[col] = series_df[col].astype(np.float32)

print('Regularized rows:', len(series_df))
print('Any null values:', bool(series_df.isna().any().any()))
series_df.head()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(series_df['timestamp'], series_df['transactions'], lw=0.7)
axes[0].set_title('Hourly Transaction Volume')
axes[0].set_ylabel('Transactions')
axes[0].grid(alpha=0.2)

axes[1].plot(series_df['timestamp'], series_df['fraud_rate'], lw=0.7, color='tab:red')
axes[1].set_title('Hourly Fraud Rate')
axes[1].set_ylabel('Fraud Rate')
axes[1].set_xlabel('Timestamp')
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

## 3) Forecasting Configuration

Targets:

- next hour (`h=1`)
- next day (`h=24`)
- next week (`h=168`)

Config:

- `context_len=1008` hours (~6 weeks history)
- `max_horizon=168` hours (1 week forecast)
- rolling anchors for robust backtesting


In [ ]:
@dataclass
class Config:
    context_len: int = 1008
    max_horizon: int = 168
    eval_horizons: tuple[int, ...] = (1, 24, 168)
    anchors_weeks_ago: tuple[int, ...] = (24, 12, 8, 4, 2)
    per_core_batch_size: int = 8
    xreg_mode: str = 'xreg + timesfm'
    xreg_ridge: float = 1e-3


cfg = Config()
arr = series_df['transactions'].to_numpy(np.float32)

anchor_positions: list[int] = []
last_pos = len(arr) - 1
for weeks_ago in cfg.anchors_weeks_ago:
    end_pos = last_pos - weeks_ago * 168
    if end_pos - cfg.context_len + 1 >= 0 and end_pos + cfg.max_horizon < len(arr):
        anchor_positions.append(end_pos)

if not anchor_positions:
    raise RuntimeError('No valid anchors found. Reduce context/horizon or use longer history.')

print('Valid anchors:', len(anchor_positions))
for pos in anchor_positions:
    print(' -', series_df['timestamp'].iloc[pos])

## 4) Load TimesFM + Feature Helpers

We include calendar and fraud-load covariates:

- hour-of-day cyclical (`sin/cos`)
- day-of-week cyclical (`sin/cos`)
- weekend flag
- `fraud_rate_proxy` (hourly fraud pressure signal)


In [ ]:
import timesfm

model = timesfm.TimesFM_2p5_200M_torch.from_pretrained('google/timesfm-2.5-200m-pytorch')
forecast_config = timesfm.ForecastConfig(
    max_context=cfg.context_len,
    max_horizon=cfg.max_horizon,
    normalize_inputs=True,
    per_core_batch_size=cfg.per_core_batch_size,
    use_continuous_quantile_head=True,
    force_flip_invariance=True,
    infer_is_positive=True,
    fix_quantile_crossing=True,
    return_backcast=True,
)
model.compile(forecast_config)
print('TimesFM compiled.')

In [ ]:
series_idx = series_df.set_index('timestamp')


def wmape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.abs(y_true - y_pred).sum() / (np.abs(y_true).sum() + 1e-8))


def build_covariates(start_ts: pd.Timestamp, context_len: int, horizon: int) -> dict[str, list[list[float]]]:
    idx = pd.date_range(start_ts, periods=context_len + horizon, freq='h')
    hour = idx.hour.values.astype(np.float32)
    dow = idx.dayofweek.values.astype(np.float32)
    fraud_rate = series_idx.reindex(idx)['fraud_rate'].ffill().fillna(0.0).astype(np.float32).values

    return {
        'hour_sin': [np.sin(2 * np.pi * hour / 24.0).astype(np.float32).tolist()],
        'hour_cos': [np.cos(2 * np.pi * hour / 24.0).astype(np.float32).tolist()],
        'dow_sin': [np.sin(2 * np.pi * dow / 7.0).astype(np.float32).tolist()],
        'dow_cos': [np.cos(2 * np.pi * dow / 7.0).astype(np.float32).tolist()],
        'is_weekend': [(dow >= 5).astype(np.float32).tolist()],
        'fraud_rate_proxy': [fraud_rate.tolist()],
    }


def run_timesfm_forecast(context: np.ndarray, start_ts: pd.Timestamp, horizon: int) -> tuple[np.ndarray, np.ndarray]:
    cov = build_covariates(start_ts=start_ts, context_len=len(context), horizon=horizon)

    try:
        point, quant = model.forecast_with_covariates(
            inputs=[context.astype(np.float32)],
            dynamic_numerical_covariates=cov,
            xreg_mode=cfg.xreg_mode,
            ridge=cfg.xreg_ridge,
        )
    except Exception:
        point, quant = model.forecast(horizon=horizon, inputs=[context.astype(np.float32)])

    p = np.asarray(point, dtype=np.float32)[0, :horizon]
    q = np.asarray(quant, dtype=np.float32)[0, :horizon, :]

    return np.clip(p, 0.0, None), np.clip(q, 0.0, None)

## 5) Rolling Backtest: TimesFM vs Baselines

Baselines:

- `naive_last`
- `seasonal24`
- `seasonal168`


In [ ]:
metric_rows: list[dict] = []

for anchor_pos in anchor_positions:
    context = arr[anchor_pos - cfg.context_len + 1 : anchor_pos + 1]
    future = arr[anchor_pos + 1 : anchor_pos + cfg.max_horizon + 1]

    start_ts = series_df['timestamp'].iloc[anchor_pos - cfg.context_len + 1]
    tfm_pred, tfm_quant = run_timesfm_forecast(context=context, start_ts=start_ts, horizon=cfg.max_horizon)

    naive_last = np.repeat(context[-1], cfg.max_horizon)
    seasonal24 = np.tile(context[-24:], math.ceil(cfg.max_horizon / 24))[: cfg.max_horizon]
    seasonal168 = np.tile(context[-168:], math.ceil(cfg.max_horizon / 168))[: cfg.max_horizon]

    pred_map = {
        'timesfm': tfm_pred,
        'naive_last': naive_last,
        'seasonal24': seasonal24,
        'seasonal168': seasonal168,
    }

    anchor_ts = series_df['timestamp'].iloc[anchor_pos]
    for hz in cfg.eval_horizons:
        y = future[:hz]
        for model_name, pred in pred_map.items():
            yhat = pred[:hz]
            metric_rows.append(
                {
                    'anchor_ts': anchor_ts,
                    'horizon_hours': hz,
                    'model': model_name,
                    'mae': mean_absolute_error(y, yhat),
                    'rmse': mean_squared_error(y, yhat) ** 0.5,
                    'wmape': wmape(y, yhat),
                }
            )

backtest_detail = pd.DataFrame(metric_rows)
backtest_metrics = (
    backtest_detail
    .groupby(['horizon_hours', 'model'], as_index=False)[['mae', 'rmse', 'wmape']]
    .mean()
    .sort_values(['horizon_hours', 'rmse'])
)

backtest_metrics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ['mae', 'rmse', 'wmape']):
    chart_df = backtest_metrics.pivot(index='horizon_hours', columns='model', values=metric)
    chart_df.plot(kind='bar', ax=ax)
    ax.set_title(metric.upper())
    ax.set_xlabel('Horizon (hours)')
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

## 6) Forecast Next 168 Hours (1 Week)

Forecast the next week from the latest available transaction history.


In [ ]:
latest_pos = len(arr) - 1
latest_ts = series_df['timestamp'].iloc[latest_pos]

context = arr[-cfg.context_len:]
context_start = series_df['timestamp'].iloc[len(arr) - cfg.context_len]

future_p50, future_quant = run_timesfm_forecast(context=context, start_ts=context_start, horizon=cfg.max_horizon)
future_index = pd.date_range(latest_ts + pd.Timedelta(hours=1), periods=cfg.max_horizon, freq='h')

forecast_df = pd.DataFrame(
    {
        'timestamp': future_index,
        'timesfm_p50_transactions': future_p50,
        'timesfm_q10_transactions': future_quant[:, 1],
        'timesfm_q90_transactions': future_quant[:, 9],
    }
)
forecast_df['uncertainty_band'] = forecast_df['timesfm_q90_transactions'] - forecast_df['timesfm_q10_transactions']

forecast_df.head(12)

## 7) Operations Layer: Capacity + Fraud-Monitoring + Reliability

We map demand forecasts into concrete planning actions.

Assumptions (replace with your platform telemetry in production):

- One app node can process `450` transactions/hour at target SLO.
- Safety utilization target: `70%`.
- One fraud analyst can review `180` transactions/hour of suspicious workload.


In [ ]:
TX_PER_NODE_PER_HOUR = 450.0
TARGET_UTILIZATION = 0.70
TX_PER_FRAUD_ANALYST_PER_HOUR = 180.0

ops_df = forecast_df.copy()

# Infra capacity planning (risk-aware by q90 demand).
ops_df['app_nodes_needed_q90'] = np.ceil(
    ops_df['timesfm_q90_transactions'] / (TX_PER_NODE_PER_HOUR * TARGET_UTILIZATION)
).astype(int)
ops_df['app_nodes_needed_q90'] = ops_df['app_nodes_needed_q90'].clip(lower=1)

# Fraud monitoring capacity planning (using upper demand bound).
ops_df['fraud_analysts_needed_q90'] = np.ceil(
    ops_df['timesfm_q90_transactions'] / TX_PER_FRAUD_ANALYST_PER_HOUR
).astype(int)
ops_df['fraud_analysts_needed_q90'] = ops_df['fraud_analysts_needed_q90'].clip(lower=1)

# Expected utilization and reliability risk if q90 sizing is followed.
ops_df['expected_utilization_p50'] = (
    ops_df['timesfm_p50_transactions']
    / (ops_df['app_nodes_needed_q90'] * TX_PER_NODE_PER_HOUR + 1e-6)
)

ops_df['estimated_p95_latency_ms'] = 90.0 + 220.0 * np.power(ops_df['expected_utilization_p50'], 2.1)
ops_df['downtime_risk_flag'] = np.select(
    [ops_df['expected_utilization_p50'] >= 0.90, ops_df['expected_utilization_p50'] >= 0.75],
    ['high', 'medium'],
    default='low',
)

# Relative load trend to aid staffing shifts.
recent_7d_avg = float(series_df['transactions'].tail(24 * 7).mean())
ops_df['load_index_vs_recent_7d'] = (
    (ops_df['timesfm_p50_transactions'] - recent_7d_avg) / (recent_7d_avg + 1e-6)
)
ops_df['resource_action'] = np.select(
    [ops_df['load_index_vs_recent_7d'] >= 0.10, ops_df['load_index_vs_recent_7d'] <= -0.10],
    ['increase_shift_coverage', 'decrease_shift_coverage'],
    default='hold_shift_coverage',
)

ops_df.head(12)

In [ ]:
kpi_summary = pd.DataFrame(
    {
        'metric': [
            'forecast_horizon_hours',
            'avg_p50_transactions_per_hour',
            'avg_q90_transactions_per_hour',
            'max_q90_transactions_per_hour',
            'avg_app_nodes_needed_q90',
            'max_app_nodes_needed_q90',
            'avg_fraud_analysts_needed_q90',
            'max_fraud_analysts_needed_q90',
            'high_risk_hours',
            'avg_estimated_p95_latency_ms',
            'max_estimated_p95_latency_ms',
        ],
        'value': [
            len(ops_df),
            float(ops_df['timesfm_p50_transactions'].mean()),
            float(ops_df['timesfm_q90_transactions'].mean()),
            float(ops_df['timesfm_q90_transactions'].max()),
            float(ops_df['app_nodes_needed_q90'].mean()),
            float(ops_df['app_nodes_needed_q90'].max()),
            float(ops_df['fraud_analysts_needed_q90'].mean()),
            float(ops_df['fraud_analysts_needed_q90'].max()),
            int((ops_df['downtime_risk_flag'] == 'high').sum()),
            float(ops_df['estimated_p95_latency_ms'].mean()),
            float(ops_df['estimated_p95_latency_ms'].max()),
        ],
    }
)

kpi_summary

## 8) Save Deliverables

Outputs saved to `artifacts/financial_transaction_volume_timesfm/`:

- `backtest_metrics.csv`
- `backtest_detail.csv`
- `forecast_next_168_hours.csv`
- `capacity_resource_plan.csv`
- `kpi_summary.csv`


In [ ]:
backtest_metrics_path = ART_DIR / 'backtest_metrics.csv'
backtest_detail_path = ART_DIR / 'backtest_detail.csv'
forecast_path = ART_DIR / 'forecast_next_168_hours.csv'
ops_path = ART_DIR / 'capacity_resource_plan.csv'
kpi_path = ART_DIR / 'kpi_summary.csv'

backtest_metrics.to_csv(backtest_metrics_path, index=False)
backtest_detail.to_csv(backtest_detail_path, index=False)
forecast_df.to_csv(forecast_path, index=False)
ops_df.to_csv(ops_path, index=False)
kpi_summary.to_csv(kpi_path, index=False)

print('Saved artifacts:')
for p in [backtest_metrics_path, backtest_detail_path, forecast_path, ops_path, kpi_path]:
    print('-', p)

backtest_metrics

## Assumptions, Limitations, and Production Next Steps

### Assumptions

- Transaction count is the primary capacity driver.
- Resource conversion coefficients are fixed and linear.

### Limitations

- No explicit campaign, holiday, or release-event drivers.
- Fraud-monitoring load is approximated from volume, not case complexity.
- Single global stream; no region/product/merchant breakdown.

### Production next steps

- Add exogenous covariates: release calendar, promotions, payroll cycles, incidents.
- Forecast at multiple slices (region, payment rail, merchant segment).
- Couple forecasts with queueing/cost optimization for autoscaling policy.
- Monitor drift and decision KPIs (SLO breaches, fraud-review SLA, cost per txn).
